# Bronze Layer Ingestion - Virtue Foundation Ghana

**Purpose:** Ingest raw CSV data into Delta Lake Bronze table

**Dataset:** Virtue Foundation Ghana v0.3 - ~978 healthcare facilities and NGOs

**Handles:**
* JSON-stringified arrays (specialties, procedure, equipment, capability, phone_numbers, websites)
* Mixed null representations (empty strings, empty arrays, nulls)
* Schema evolution and column renaming
* Comprehensive data quality validation

**Output Table:** `virtue_foundation.ghana.facilities_bronze`

## 1. Setup and Configuration

Import required libraries and define configuration variables.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import json

# Configuration
CATALOG = "virtue_foundation"
SCHEMA = "ghana"
TABLE_NAME = "facilities_bronze"

# CSV file path - using actual filename with spaces
CSV_PATH = "/Workspace/Users/anuragrc27@gmail.com/Databricks-AI-Agent/data/Virtue Foundation Ghana v0.3 - Sheet1.csv"

# Alternative paths (uncomment if needed):
# CSV_PATH = "/Volumes/virtue_foundation/ghana/raw_data/Virtue_Foundation_Ghana_v0_3_-_Sheet1.csv"
# CSV_PATH = "dbfs:/FileStore/tables/Virtue_Foundation_Ghana_v0_3_-_Sheet1.csv"

print(f"Target table: {CATALOG}.{SCHEMA}.{TABLE_NAME}")
print(f"Source CSV: {CSV_PATH}")
print(f"Today's date: 2026-04-12")

In [0]:
%sql
-- Create catalog and schema if they don't exist
CREATE CATALOG IF NOT EXISTS virtue_foundation;
CREATE SCHEMA IF NOT EXISTS virtue_foundation.ghana;

USE CATALOG virtue_foundation;
USE SCHEMA ghana;

SHOW TABLES;

## 2. Read CSV with Proper Configuration

Load CSV with options to handle:
* Multi-line fields
* Quoted values with commas
* JSON arrays in columns
* Schema inference

In [0]:
# Source: Workspace file path (not readable by Spark)
workspace_path = "/Workspace/Users/anuragrc27@gmail.com/Databricks-AI-Agent/data/Virtue Foundation Ghana v0.3 - Sheet1.csv"

# Create volume if it doesn't exist
spark.sql("""
    CREATE VOLUME IF NOT EXISTS virtue_foundation.ghana.raw_data
""")

# Destination: Volumes path (readable by Spark)
volumes_path = "/Volumes/virtue_foundation/ghana/raw_data/Virtue_Foundation_Ghana_v0_3.csv"

# Copy file using dbutils (handles Workspace to Volumes copy)
print(f"Copying CSV from Workspace to Volumes...")
dbutils.fs.cp(f"file:{workspace_path}", volumes_path, recurse=False)
print(f"✅ File copied to: {volumes_path}")

# Now read from Volumes path
try:
    df_raw = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .option("multiLine", "true") \
        .option("escape", '"') \
        .option("quote", '"') \
        .option("mode", "PERMISSIVE") \
        .load(volumes_path)
    
    row_count = df_raw.count()
    col_count = len(df_raw.columns)
    
    print(f"\n✅ CSV loaded successfully")
    print(f"Total rows: {row_count:,}")
    print(f"Total columns: {col_count}")
    print(f"\nColumns: {df_raw.columns[:10]}... (showing first 10)")
    
    # Show sample
    print("\nSample data (first 3 rows):")
    display(df_raw.limit(3))
    
except Exception as e:
    print(f"❌ Error loading CSV: {e}")
    print(f"\nTroubleshooting:")
    print(f"1. Verify file exists at: {volumes_path}")
    print(f"2. Check file permissions")
    print(f"3. Verify CSV format (UTF-8, proper headers)")
    raise

## 3. Data Cleaning and Transformation

Apply initial transformations:
* Rename columns with spaces
* Add ingestion metadata
* Create record hash for deduplication

In [0]:
# Rename column with space if it exists
if "mongo DB" in df_raw.columns:
    df_raw = df_raw.withColumnRenamed("mongo DB", "mongo_db_id")
    print("✅ Renamed 'mongo DB' to 'mongo_db_id'")
else:
    print("ℹ️  Column 'mongo DB' not found, skipping rename")

# Check for other problematic column names
problematic_cols = [col for col in df_raw.columns if ' ' in col or '-' in col]
if problematic_cols:
    print(f"⚠️  Found columns with spaces/dashes: {problematic_cols}")
    for col in problematic_cols:
        new_col = col.replace(' ', '_').replace('-', '_')
        df_raw = df_raw.withColumnRenamed(col, new_col)
        print(f"   Renamed '{col}' to '{new_col}'")

In [0]:
# Add metadata columns for tracking
df_bronze = df_raw \
    .withColumn("ingestion_timestamp", F.current_timestamp()) \
    .withColumn("source_file", F.lit(CSV_PATH)) \
    .withColumn("record_hash", F.md5(F.concat_ws("||", *df_raw.columns)))

print(f"✅ Added metadata columns:")
print(f"   • ingestion_timestamp - Current timestamp for audit trail")
print(f"   • source_file - Source CSV path")
print(f"   • record_hash - MD5 hash for deduplication")
print(f"\nTotal columns after metadata: {len(df_bronze.columns)}")

## 4. Write to Delta Lake Bronze Table

Write data to Delta Lake with:
* Overwrite mode (idempotent ingestion)
* Schema evolution enabled
* Optimization for future queries

In [0]:
# Write as Delta table with schema evolution
full_table_name = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"

try:
    df_bronze.write.format("delta") \
        .mode("overwrite") \
        .option("mergeSchema", "true") \
        .option("overwriteSchema", "true") \
        .saveAsTable(full_table_name)
    
    print(f"✅ Bronze table created successfully: {full_table_name}")
    print(f"   Mode: OVERWRITE (full refresh)")
    print(f"   Schema evolution: ENABLED")
    print(f"   Rows written: {df_bronze.count():,}")
    
except Exception as e:
    print(f"❌ Error writing to Delta table: {e}")
    print(f"\nTroubleshooting:")
    print(f"1. Check Unity Catalog permissions")
    print(f"2. Verify catalog and schema exist")
    print(f"3. Check storage location permissions")
    raise

In [0]:
%sql
-- Add descriptive comment to table
COMMENT ON TABLE virtue_foundation.ghana.facilities_bronze IS 
'Raw ingestion of Virtue Foundation Ghana v0.3 facility dataset. 
Contains ~978 healthcare facilities and NGOs with structured and semi-structured data.
JSON arrays in columns: specialties, procedure, equipment, capability, phone_numbers, websites, affiliationTypeIds.
Ingested on 2026-04-12.';

-- Set table properties
ALTER TABLE virtue_foundation.ghana.facilities_bronze 
SET TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true',
  'source' = 'virtue_foundation_csv',
  'layer' = 'bronze',
  'quality' = 'raw'
);

## 5. Verification and Data Quality Checks

Verify ingestion success:
* Row count validation
* Schema inspection
* Sample data review
* Null value analysis
* Distribution checks

In [0]:
# Read back table to verify
df_verify = spark.table(f"{CATALOG}.{SCHEMA}.{TABLE_NAME}")

print("="*80)
print("INGESTION SUMMARY")
print("="*80)
print(f"Table: {CATALOG}.{SCHEMA}.{TABLE_NAME}")
print(f"Total rows: {df_verify.count():,}")
print(f"Total columns: {len(df_verify.columns)}")
print(f"Ingestion time: {df_verify.select(F.max('ingestion_timestamp')).collect()[0][0]}")
print("="*80)

# Verify expected row count
expected_rows = 978
actual_rows = df_verify.count()
if actual_rows >= expected_rows * 0.95:  # Within 5% of expected
    print(f"✅ Row count validation PASSED: {actual_rows} rows (expected ~{expected_rows})")
else:
    print(f"⚠️  Row count warning: {actual_rows} rows (expected ~{expected_rows})")

In [0]:
# Show schema with data types
print("\nTABLE SCHEMA:")
print("="*80)
df_verify.printSchema()

# Show column summary
print("\nKEY COLUMNS:")
key_cols = ['name', 'facilityTypeId', 'operatorTypeId', 'address_city', 
            'address_stateOrRegion', 'numberDoctors', 'capacity', 'specialties', 
            'procedure', 'equipment']
existing_key_cols = [col for col in key_cols if col in df_verify.columns]
for col in existing_key_cols:
    dtype = dict(df_verify.dtypes)[col]
    print(f"  • {col:30s} ({dtype})")

In [0]:
# Display sample records with key columns
print("\nSAMPLE RECORDS (10 rows):")
print("="*80)

sample_cols = ['name', 'facilityTypeId', 'address_city', 'address_stateOrRegion', 
               'numberDoctors', 'capacity', 'specialties', 'procedure', 'equipment']
existing_sample_cols = [col for col in sample_cols if col in df_verify.columns]

display(df_verify.select(*existing_sample_cols).limit(10))

## 6. Data Quality Analysis

Comprehensive quality checks:
* Null value analysis by column
* Facility type distribution
* Regional coverage
* JSON array column inspection

In [0]:
# Check for null values in all columns
print("NULL VALUE ANALYSIS (Top 20 columns by null count):")
print("="*80)

null_counts = []
for col_name in df_verify.columns:
    null_count = df_verify.filter(F.col(col_name).isNull()).count()
    null_pct = (null_count / df_verify.count()) * 100
    null_counts.append((col_name, null_count, null_pct))

# Sort by null count descending
null_counts_sorted = sorted(null_counts, key=lambda x: x[1], reverse=True)[:20]

for col, count, pct in null_counts_sorted:
    bar = '█' * min(int(pct / 2), 50)  # Visual bar
    print(f"{col:35s}: {count:4d} nulls ({pct:5.1f}%) {bar}")

# Flag critical columns with high null rates
critical_cols = ['name', 'facilityTypeId', 'address_stateOrRegion']
print(f"\nCRITICAL COLUMN CHECK:")
for col in critical_cols:
    if col in df_verify.columns:
        null_count = df_verify.filter(F.col(col).isNull()).count()
        if null_count > 0:
            print(f"⚠️  {col}: {null_count} nulls found!")
        else:
            print(f"✅ {col}: No nulls")

In [0]:
# Check facility type distribution
print("\nFACILITY TYPE DISTRIBUTION:")
print("="*80)

if 'facilityTypeId' in df_verify.columns:
    facility_dist = df_verify.groupBy("facilityTypeId") \
        .count() \
        .orderBy(F.desc("count"))
    
    display(facility_dist)
    
    # Summary stats
    total = df_verify.count()
    types = facility_dist.count()
    print(f"\nTotal facility types: {types}")
    print(f"Most common type: {facility_dist.first()['facilityTypeId']} ({facility_dist.first()['count']} facilities)")
else:
    print("⚠️  Column 'facilityTypeId' not found")

In [0]:
# Check region distribution
print("\nREGIONAL DISTRIBUTION:")
print("="*80)

if 'address_stateOrRegion' in df_verify.columns:
    region_dist = df_verify.groupBy("address_stateOrRegion") \
        .agg(
            F.count("*").alias("total_facilities"),
            F.count_distinct("address_city").alias("cities_covered")
        ) \
        .orderBy(F.desc("total_facilities"))
    
    display(region_dist)
    
    # Regional stats
    total_regions = region_dist.count()
    print(f"\nTotal regions: {total_regions}")
    print(f"Facilities span {total_regions} administrative regions")
    
    # Check for potential medical deserts (regions with <5 facilities)
    low_coverage = region_dist.filter(F.col("total_facilities") < 5).count()
    if low_coverage > 0:
        print(f"⚠️  {low_coverage} regions have fewer than 5 facilities (potential medical deserts)")
else:
    print("⚠️  Column 'address_stateOrRegion' not found")

In [0]:
# Inspect JSON array columns
print("\nJSON ARRAY COLUMNS INSPECTION:")
print("="*80)

json_cols = ['specialties', 'procedure', 'equipment', 'capability', 'phone_numbers', 'websites']
existing_json_cols = [col for col in json_cols if col in df_verify.columns]

if existing_json_cols:
    # Show examples of JSON array columns
    print(f"\nFound {len(existing_json_cols)} JSON array columns: {existing_json_cols}")
    print(f"\nSample records with populated JSON arrays:")
    
    # Filter for rows with non-null values in at least one JSON column
    filter_condition = None
    for col in existing_json_cols[:3]:  # Check first 3
        if filter_condition is None:
            filter_condition = F.col(col).isNotNull()
        else:
            filter_condition = filter_condition | F.col(col).isNotNull()
    
    if filter_condition is not None:
        sample_json = df_verify.select(
            "name",
            *existing_json_cols
        ).filter(filter_condition).limit(5)
        
        display(sample_json)
    
    # Count non-null values per JSON column
    print(f"\nJSON ARRAY POPULATION:")
    for col in existing_json_cols:
        non_null = df_verify.filter(F.col(col).isNotNull()).count()
        pct = (non_null / df_verify.count()) * 100
        print(f"{col:20s}: {non_null:4d} populated ({pct:5.1f}%)")
else:
    print("ℹ️  No JSON array columns found with expected names")

## ✅ Bronze Ingestion Complete!

**What's been created:**
* ✅ Delta table: `virtue_foundation.ghana.facilities_bronze`
* ✅ Schema with all original columns + metadata
* ✅ Data quality metrics and validation
* ✅ Ready for Silver layer transformation

**Next Steps:**
1. **Review data quality findings** above
2. **Run Silver DLT pipeline** (`02_Silver_Transformation_DLT`) to:
   * Parse JSON arrays into proper ArrayType columns
   * Normalize null values
   * Add computed columns (completeness_score, has_procedures, etc.)
   * Apply data quality expectations
3. **Create Gold aggregations** (`03_Gold_Regional_Summary`)
4. **Setup Genie** for natural language queries

**Troubleshooting:**
* If row count is off, check CSV file integrity
* If regions show unexpected nulls, verify CSV encoding (UTF-8)
* If JSON columns aren't parsing, they'll be fixed in Silver layer